# Phase 7: Evidence-Backed Underwriting Review Package

The last phase converts structured workflow state into a package for a qualified human underwriter. It never returns `approved` or `denied`; it returns a recommendation, evidence, rules, exceptions, conditions, and an attributable activity log. The internal field remains `review_disposition` for API compatibility.

In [ ]:
# Locate the src-layout project and import the completed earlier phases.
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "underwritingAgent", Path.cwd().parent / "underwritingAgent"]
PROJECT_ROOT = next(path.resolve() for path in candidates if (path / "pyproject.toml").exists())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PDF_ROOT = PROJECT_ROOT / "data" / "realistic_pdfs"
GUIDELINES = PROJECT_ROOT / "data" / "underwriting_guidelines.jsonl"
INTAKE_REFERENCES = ["UW-26-0417-A", "BRK-90831", "WHL-77-2206"]
print(PROJECT_ROOT)


## 1. Run the complete Phase 2→7 pipeline

In [ ]:
from underwriting_agent.pipeline import run_underwriting_pipeline

def run(loan_id):
    return run_underwriting_pipeline(loan_id, PDF_ROOT, GUIDELINES)

clean = run("UW-26-0417-A")
clean["review_package"].model_dump(mode="json")


## 2. Inspect the showcase review package

The Nguyen package combines mixed income, a deposit requiring documentation, and a low appraisal. The package escalates those facts; a human makes the final decision.

In [ ]:
showcase = run("WHL-77-2206")["review_package"]
print(showcase.executive_summary)
print("\nRules:", showcase.applicable_rule_ids)
print("\nConditions:")
for condition in showcase.conditions:
    print(" -", condition)
print("\n", showcase.disclaimer)


## 3. Evaluate final recommendations for all packages

In [ ]:
portfolio = []
for loan_id in INTAKE_REFERENCES:
    package = run(loan_id)["review_package"]
    portfolio.append({"intake_reference": loan_id, "recommendation": package.review_disposition, "exceptions": [item.code for item in package.exceptions], "human_review_required": package.human_review_required})
portfolio


## 4. Activity log and UI integration boundary

`observability_log` identifies each action as `ai` or `human`, including uploads and exception-review responses. `UnderwritingReviewPackage` is the Streamlit contract, so the UI renders facts, provenance, citations, conditions, research, and activity without parsing prose.

In [ ]:
for event in showcase.observability_log:
    print(event.timestamp, event.actor, event.phase, "-", event.action)


## Optional production services: OpenAI and Pinecone

`OpenAIReviewNarrator` can turn already-verified facts, retrieved rule IDs, exceptions, and conditions into clearer prose using strict structured output. It cannot add facts, perform calculations, or make an approval/denial recommendation. The deterministic summary remains the offline default and the structured review package remains the UI contract.

In [ ]:
import os
from dotenv import load_dotenv
from underwriting_agent.model_services import OpenAIReviewNarrator
from underwriting_agent.summary import build_summary_workflow

load_dotenv(PROJECT_ROOT / ".env", override=False)
if os.getenv("USE_OPENAI_REVIEW_NARRATOR", "false").lower() == "true":
    narrator = OpenAIReviewNarrator(os.getenv("OPENAI_MODEL", "gpt-4o-mini"))
    model_summary_workflow = build_summary_workflow(narrator=narrator)
    print("OpenAI review narrative enabled")
else:
    print("Deterministic review narrative enabled (offline default)")
